# Algorithm 20 seeded KLDM-PPR from oracle space group

This notebook tests the seeded-basin version of Algorithm 20.

We assume CSPML gives the correct space group `W` (treated as oracle for now), then:

1. generate cheap PyXtal candidates for `(A, W)`
2. turn each candidate into a seed payload/template surrogate
3. score candidates cheaply before reverse sampling
4. select the top seeds
5. forward-noise those seeds with native KLDM/TDM
6. run baseline reverse and Algorithm 20 q-witness PPR as a **local refiner**

The goal here is not “discover symmetry from a free KLDM prior sample.”
The goal is to prove the seeded constrained basin story first.


In [ ]:
import os
import math
import random
import traceback
import importlib
from collections import Counter
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
import kldmPlus.algorithm20_kldm_ppr_q_witness as algo20_backend
import kldmPlus.algorithm18_hard_fractional_kldm_ppr as algo18_backend
algo19_backend = importlib.reload(algo19_backend)
algo20_backend = importlib.reload(algo20_backend)
algo18_backend = importlib.reload(algo18_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import (
    build_symmetry_frame_bridge,
    estimate_semantic_transport_for_reference_order,
    oracle_spacegroup_from_case,
)
from kldmPlus.symmetry.pyxtal_backend import build_pyxtal_wyckoff_result
from kldmPlus.symmetry.wyckoff_templates import expand_wyckoff_template_torch
from kldmPlus.algorithm12_lattice_casal_kspace import cell_to_l
from kldmPlus.utils.time import iter_sampling_times

Algorithm19State = algo19_backend.Algorithm19State
Algorithm20Config = algo20_backend.Algorithm20Config
q_only_witness_fit = algo20_backend.q_only_witness_fit
ppr_kernel_q_witness = algo20_backend.ppr_kernel_q_witness
witness_torus_sin_loss = algo20_backend.witness_torus_sin_loss
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
map_model_to_payload_reference_chart = algo19_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo19_backend.map_payload_reference_chart_to_model_frame
torus_rmse = algo19_backend.torus_rmse
wrapdiff = algo19_backend.wrapdiff
wrap01 = algo19_backend.wrap01
build_template_and_free_vars_from_pyxtal = algo18_backend.build_template_and_free_vars_from_pyxtal

_cwd = Path.cwd().resolve()
if (_cwd / 'configs').exists() and (_cwd / 'src').exists():
    ROOT = _cwd
elif (_cwd.parent / 'configs').exists() and (_cwd.parent / 'src').exists():
    ROOT = _cwd.parent
else:
    ROOT = _cwd
CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO20_SEEDED_PROFILE', 'laptop').strip().lower()

def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO20_SEEDED_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO20_SEEDED_GRAPH_IDS', '2', '2'))
PYXTAL_NUM_CANDIDATES = int(profile_default('KLDM_ALGO20_PYXTAL_NUM_CANDIDATES', '20', '24'))
PYXTAL_MAX_ATTEMPTS = int(profile_default('KLDM_ALGO20_PYXTAL_MAX_ATTEMPTS', '120', '240'))
PYXTAL_TOP_K = int(profile_default('KLDM_ALGO20_PYXTAL_TOP_K', '3', '3'))
PYXTAL_T_PROBE = float(profile_default('KLDM_ALGO20_PYXTAL_T_PROBE', '0.5', '0.5'))
ALGO20_Q_ONLY_STEPS = int(profile_default('KLDM_ALGO20_Q_ONLY_STEPS', '100', '150'))
ALGO20_PROJ_STEPS = int(profile_default('KLDM_ALGO20_PROJ_STEPS', '100', '150'))
ALGO20_LR = float(profile_default('KLDM_ALGO20_LR', '1e-2', '1e-2'))
ALGO20_LAMBDA_FLOOR = float(profile_default('KLDM_ALGO20_LAMBDA_FLOOR', '1e-6', '1e-6'))
ALGO20_GRAD_CLIP = float(profile_default('KLDM_ALGO20_GRAD_CLIP', '10.0', '10.0'))
ALGO20_SOFT_ANCHOR_TOL = float(profile_default('KLDM_ALGO20_SOFT_ANCHOR_TOL', '1e-5', '1e-5'))
ALGO20_DENOISER_VARIANT = str(profile_default('KLDM_ALGO20_DENOISER_VARIANT', 'minus', 'minus'))
ALGO20_COORDINATE_SCORE_MODE = str(profile_default('KLDM_ALGO20_COORDINATE_SCORE_MODE', 'direct', 'direct'))
SEEDED_FULL_STEPS = int(profile_default('KLDM_ALGO20_SEEDED_FULL_STEPS', '300', '600'))
SEEDED_PROJECTION_INTERVAL = int(profile_default('KLDM_ALGO20_SEEDED_PROJECTION_INTERVAL', '50', '50'))
SEEDED_PPR_M = int(profile_default('KLDM_ALGO20_SEEDED_M', '2', '2'))
SEEDED_PPR_LAMBDA0 = float(profile_default('KLDM_ALGO20_SEEDED_LAMBDA0', '1.0', '1.0'))
SEEDED_ANCHOR_MODE = str(profile_default('KLDM_ALGO20_SEEDED_ANCHOR_MODE', 'soft', 'soft'))
SEEDED_TAU = float(profile_default('KLDM_ALGO20_SEEDED_TAU', '0.25', '0.25'))
SEEDED_ROLLBACK = str(profile_default('KLDM_ALGO20_SEEDED_ROLLBACK', 'true', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}

runner = SamplingCompareRunner(CONFIG)
runner.setup()
set_seed(SAMPLE_SEED)

batch = runner.batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()
available_graph_ids = [int(v) for v in torch.as_tensor(batch.graph_id).reshape(-1).tolist()]

print(f'profile={PROFILE} graphs={GRAPH_IDS} pyxtal_candidates={PYXTAL_NUM_CANDIDATES} top_k={PYXTAL_TOP_K}')
print(f'seeded_steps={SEEDED_FULL_STEPS} projection_interval={SEEDED_PROJECTION_INTERVAL} M={SEEDED_PPR_M} lambda0={SEEDED_PPR_LAMBDA0}')


In [ ]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


result_tables: dict[str, pd.DataFrame] = {}
seed_payload_cache: dict[tuple[int, str], Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = (
        (edge_index[0] >= start)
        & (edge_index[0] < end)
        & (edge_index[1] >= start)
        & (edge_index[1] < end)
    )
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
        'num_atoms': int(end - start),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        if int(graph_id) not in set(int(v) for v in GRAPH_IDS):
            continue
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(pos=pos, batch=batch_index, num_atoms=num_atoms, edge_node_index=edge_node_index)


def single_graph_batch(case: GraphCase):
    selected = batch.index_select([int(case.graph_idx0)]) if hasattr(batch, 'index_select') else batch[int(case.graph_idx0)]
    if isinstance(selected, list):
        selected = batch.__class__.from_data_list(selected)
    return selected.to(runner.device)


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(
        case.gt_frac,
        case.gt_l_feature,
        case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )


def algo_times(num_atoms: int, t_value: float, *, device, dtype):
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(num_atoms),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(
    *,
    f: torch.Tensor,
    v: torch.Tensor,
    l: torch.Tensor,
    atom_types: torch.Tensor,
    node_index: torch.Tensor,
    edge_node_index: torch.Tensor,
    t_graph: torch.Tensor,
    t_nodes: torch.Tensor,
) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State) -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=ALGO20_DENOISER_VARIANT,
        coordinate_score_mode=ALGO20_COORDINATE_SCORE_MODE,
    )


def witness_stats_from_q(z_payload: torch.Tensor, payload, q: torch.Tensor) -> dict[str, float]:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q.reshape(-1), 1.0) if q.numel() else q.reshape(-1)
    z_sym = chart.expand_q(q_eval, device=z_payload.device, dtype=z_payload.dtype)
    return {
        'witness_sin': float(witness_torus_sin_loss(z_payload, z_sym).detach().item()),
        'witness_rmse_payload': float(torus_rmse(z_payload, z_sym).detach().item()),
    }


def make_payload_q_init(payload, *, mode: str, seed: int, graph_id: int, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    total_dof = int(chart.total_dof)
    if total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'oracle_structure', 'chart_q_ref', 'bootstrap', 'seed'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'zero', 'zeros'}:
        return torch.zeros((total_dof,), device=device, dtype=dtype)
    if mode_norm in {'random', 'rand'}:
        set_seed(int(seed) + 30000 * int(graph_id))
        return torch.rand((total_dof,), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q init mode: {mode!r}')


def q_only_fit_from_clean(clean_f: torch.Tensor, payload, *, graph_id: int, q_init_mode: str, seed: int, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    q_seed = q_init if q_init is not None else make_payload_q_init(
        payload,
        mode=q_init_mode,
        seed=seed,
        graph_id=graph_id,
        dtype=clean_f.dtype,
        device=clean_f.device,
    )
    before = witness_stats_from_q(z_payload, payload, q_seed)
    fit = q_only_witness_fit(
        z_payload=z_payload,
        payload=payload,
        q_init=q_seed,
        q_init_mode=q_init_mode,
        steps=int(ALGO20_Q_ONLY_STEPS),
        lr=float(ALGO20_LR),
        grad_clip=float(ALGO20_GRAD_CLIP),
    )
    return {
        'z_payload': z_payload,
        'q_seed': q_seed,
        'before': before,
        'fit': fit,
    }


def config20(*, M: int, lambda0: float, q_init_mode: str = 'seed') -> Algorithm20Config:
    return Algorithm20Config(
        M=int(M),
        proj_steps=int(ALGO20_PROJ_STEPS),
        lr=float(ALGO20_LR),
        lambda_noise=float(lambda0),
        lambda_floor=float(ALGO20_LAMBDA_FLOOR),
        grad_clip=float(ALGO20_GRAD_CLIP),
        anchor_mode=str(SEEDED_ANCHOR_MODE),
        denoiser_variant=str(ALGO20_DENOISER_VARIANT),
        coordinate_score_mode=str(ALGO20_COORDINATE_SCORE_MODE),
        soft_anchor_tol=float(ALGO20_SOFT_ANCHOR_TOL),
        q_init_mode=str(q_init_mode),
        q_only_steps=int(ALGO20_Q_ONLY_STEPS),
    )


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'min_pair_distance': result.min_pair_distance,
    }


## 1. PyXtal seed generation and payload construction

This section keeps the space group oracle but removes the GT coordinates from the **proposal** stage.
We use PyXtal to generate symmetry-compatible seeds for `(composition A, oracle W)`, then build seed payloads from those candidate structures.


In [ ]:
def require_pyxtal_runtime():
    try:
        from pyxtal import pyxtal as PyXtalCrystal
        return PyXtalCrystal
    except Exception as exc:
        raise ImportError(
            'PyXtal is required for this notebook. Install `pyxtal` in the notebook environment before running the seeded pipeline.'
        ) from exc


def composition_to_pyxtal_inputs(case: GraphCase):
    species = []
    counts = []
    for atomic_number, count in sorted(case.composition.items()):
        species.append(int(atomic_number))
        counts.append(int(count))
    return species, counts


def structure_min_pair_distance(structure) -> float:
    distance_matrix = np.asarray(structure.distance_matrix, dtype=float)
    if distance_matrix.shape[0] <= 1:
        return float('inf')
    upper = distance_matrix[np.triu_indices(distance_matrix.shape[0], k=1)]
    if upper.size == 0:
        return float('inf')
    return float(np.min(upper))


def build_seed_payload_from_structure(structure, *, requested_spacegroup: int):
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=int(requested_spacegroup),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(
        vanilla_structure=structure,
        standardization='refined',
        symprec=1e-2,
        angle_tolerance=5.0,
    )
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(bridge.vanilla_frac_coords, dtype=float).copy(),
            'model_to_payload_linear': linear_model_to_std.copy(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).copy(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).copy(),
            'payload_to_model_linear': linear_std_to_model.copy(),
            'payload_to_model_tau': np.asarray(-linear_std_to_model @ tau_ref, dtype=float).copy(),
            'payload_to_model_order': np.argsort(np.asarray(assignment_ref, dtype=int)),
            'payload_reference_rmse': float(rmse_ref),
            'payload_source': 'pyxtal_seed',
        },
    )
    return payload


def payload_seed_to_model_state(payload, *, num_atoms: int, atom_types: torch.Tensor, device, dtype):
    z_seed_payload = torch.as_tensor(np.asarray(payload.expanded_frac_coords, dtype=float), device=device, dtype=dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_seed_payload, payload).detach().clone()
    lattice_matrix = torch.as_tensor(np.asarray(payload.lattice_matrix, dtype=float), device=device, dtype=dtype)
    l0 = cell_to_l(
        lattice_matrix,
        num_atoms=int(num_atoms),
        lattice_transform=runner.lattice_transform,
    ).to(device=device, dtype=dtype)
    return {
        'z_seed_payload': z_seed_payload,
        'f0_model': f0_model,
        'l0': l0,
        'atom_types': atom_types.detach().clone().to(device=device),
    }


def build_pyxtal_candidate(case: GraphCase, *, candidate_idx: int, factor: float, seed: int):
    PyXtalCrystal = require_pyxtal_runtime()
    species, counts = composition_to_pyxtal_inputs(case)
    set_seed(int(seed))
    crystal = PyXtalCrystal()
    crystal.from_random(
        3,
        int(case.requested_sg),
        species=species,
        numIons=counts,
        factor=float(factor),
    )
    structure = crystal.to_pymatgen()
    pyxtal_result = build_pyxtal_wyckoff_result(structure, symprec=1e-2, pyxtal_tol=1e-2)
    payload = build_seed_payload_from_structure(structure, requested_spacegroup=case.requested_sg)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    template, free_vars, templates = build_template_and_free_vars_from_pyxtal(
        pyxtal_result=pyxtal_result,
        atomic_numbers_standardized=case.atomic_numbers,
        max_templates=64,
    )
    expansion = expand_wyckoff_template_torch(template=template, free_vars=free_vars)
    template_coords = expansion.frac_coords.squeeze(0).detach().cpu()
    pyxtal_coords = torch.as_tensor(pyxtal_result.expanded_frac_coords, dtype=template_coords.dtype)
    seed_model = payload_seed_to_model_state(
        payload,
        num_atoms=int(case.atomic_numbers.numel()),
        atom_types=case.atomic_numbers,
        device=case.gt_frac.device,
        dtype=case.gt_frac.dtype,
    )
    return {
        'candidate_id': f'g{case.graph_id}_seed{candidate_idx:02d}',
        'case': case,
        'structure': structure,
        'pyxtal_result': pyxtal_result,
        'payload': payload,
        'chart': chart,
        'template': template,
        'free_vars': free_vars.detach().clone(),
        'num_templates_total': int(len(templates)),
        'template_self_rmse': float(torus_rmse(pyxtal_coords.to(case.gt_frac.device), template_coords.to(case.gt_frac.device)).detach().item()),
        'template_self_sin': float(witness_torus_sin_loss(pyxtal_coords.to(case.gt_frac.device), template_coords.to(case.gt_frac.device)).detach().item()),
        'min_pair_distance': structure_min_pair_distance(structure),
        'volume': float(structure.lattice.volume),
        'factor': float(factor),
        **seed_model,
    }


def generate_pyxtal_candidates(case: GraphCase, *, num_candidates: int = PYXTAL_NUM_CANDIDATES, max_attempts: int = PYXTAL_MAX_ATTEMPTS, seed: int = SAMPLE_SEED):
    factors = [0.75, 0.9, 1.0, 1.1, 1.25, 1.5]
    candidates = []
    rows = []
    attempt = 0
    candidate_idx = 0
    while len(candidates) < int(num_candidates) and attempt < int(max_attempts):
        factor = factors[attempt % len(factors)]
        local_seed = int(seed) + 100000 * int(case.graph_id) + int(attempt)
        attempt += 1
        try:
            candidate = build_pyxtal_candidate(case, candidate_idx=candidate_idx, factor=factor, seed=local_seed)
            candidates.append(candidate)
            rows.append({
                'test': 'algo20_pyxtal_candidates',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'candidate_id': candidate['candidate_id'],
                'factor': float(candidate['factor']),
                'num_atoms': int(case.atomic_numbers.numel()),
                'num_template_matches': int(candidate['num_templates_total']),
                'template_self_rmse': float(candidate['template_self_rmse']),
                'template_self_sin': float(candidate['template_self_sin']),
                'min_pair_distance': float(candidate['min_pair_distance']),
                'volume': float(candidate['volume']),
                'PASS': True,
                'status': 'INFO',
            })
            candidate_idx += 1
        except Exception as exc:
            rows.append(error_row(
                'algo20_pyxtal_candidates',
                case.graph_id,
                exc,
                'pyxtal_generation',
                space_group=case.requested_sg,
                attempt=int(attempt),
                factor=float(factor),
            ))
    return candidates, rows


## 2. Cheap prior scoring

For each PyXtal seed we score:

- PyXtal validity / clash proxy
- template self-consistency
- a KLDM basin probe at `t_probe = 0.5`

The basin probe is the important one: we forward-noise the seed, denoise once, and ask whether Algorithm 20's q-only witness fit still sees the same template basin.


In [ ]:
def make_seed_probe_state(candidate, *, t_value: float = PYXTAL_T_PROBE, seed: int = SAMPLE_SEED):
    case = candidate['case']
    batch_view = make_single_graph_batch_view(case, ref_tensor=candidate['f0_model'])
    set_seed(int(seed) + 400000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(int(case.atomic_numbers.numel()), float(t_value), device=candidate['f0_model'].device, dtype=candidate['f0_model'].dtype)
    f_t, v_t, eps_v, eps_r, r_t = runner.model.tdm.sample_noisy_state(
        t=t_nodes,
        f0=candidate['f0_model'],
        index=batch_view.batch,
    )
    state = Algorithm19State(
        f=f_t.detach().clone(),
        v=v_t.detach().clone(),
        l=candidate['l0'].detach().clone(),
        atom_types=case.atomic_numbers.detach().clone(),
        node_index=batch_view.batch.detach().clone(),
        edge_node_index=batch_view.edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )
    return state, {'eps_v': eps_v.detach().clone(), 'eps_r': eps_r.detach().clone(), 'r_t': r_t.detach().clone()}


def cheap_score_candidate(candidate, *, t_probe: float = PYXTAL_T_PROBE, seed: int = SAMPLE_SEED):
    case = candidate['case']
    payload = candidate['payload']
    probe_state, _ = make_seed_probe_state(candidate, t_value=float(t_probe), seed=int(seed))
    clean_f = clean_prediction(probe_state)
    q_probe = q_only_fit_from_clean(
        clean_f,
        payload,
        graph_id=case.graph_id,
        q_init_mode='seed',
        seed=int(seed),
    )
    eval_probe = evaluate_arrays(case, clean_f, probe_state.l, case.atomic_numbers)
    basin_sin = float(q_probe['fit']['witness_sin'])
    basin_rmse = float(q_probe['fit']['witness_rmse_payload'])
    clash_penalty = 0.0 if float(candidate['min_pair_distance']) >= 0.8 else (0.8 - float(candidate['min_pair_distance']))
    total_score = float(basin_sin + 0.10 * clash_penalty + 10.0 * float(candidate['template_self_sin']))
    return {
        'test': 'algo20_seed_scoring',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'candidate_id': candidate['candidate_id'],
        't_probe': float(t_probe),
        'min_pair_distance': float(candidate['min_pair_distance']),
        'template_self_sin': float(candidate['template_self_sin']),
        'template_self_rmse': float(candidate['template_self_rmse']),
        'probe_witness_sin': basin_sin,
        'probe_witness_rmse': basin_rmse,
        'probe_frac_rmse_to_gt': float(eval_probe['frac_rmse']),
        'score_total': float(total_score),
        'PASS': bool(math.isfinite(total_score)),
        'status': 'INFO',
    }


candidate_rows = []
scoring_rows = []
pyxtal_candidates_by_graph: dict[int, list[dict[str, Any]]] = {}
selected_candidates_by_graph: dict[int, list[dict[str, Any]]] = {}
for case in GRAPH_CASES:
    try:
        candidates, rows = generate_pyxtal_candidates(case)
        candidate_rows.extend(rows)
        pyxtal_candidates_by_graph[case.graph_id] = candidates
        for candidate in candidates:
            try:
                scoring_rows.append(cheap_score_candidate(candidate))
            except Exception as exc:
                scoring_rows.append(error_row(
                    'algo20_seed_scoring',
                    case.graph_id,
                    exc,
                    'cheap_seed_scoring',
                    space_group=case.requested_sg,
                    candidate_id=candidate['candidate_id'],
                ))
        scored = [row for row in scoring_rows if row.get('graph') == case.graph_id and row.get('status') != 'ERROR']
        ranked_ids = [row['candidate_id'] for row in sorted(scored, key=lambda row: float(row['score_total']))[: int(PYXTAL_TOP_K)]]
        candidate_by_id = {cand['candidate_id']: cand for cand in candidates}
        selected_candidates_by_graph[case.graph_id] = [candidate_by_id[cid] for cid in ranked_ids if cid in candidate_by_id]
    except Exception as exc:
        candidate_rows.append(error_row(
            'algo20_pyxtal_candidates',
            case.graph_id,
            exc,
            'pyxtal_candidate_pipeline',
            space_group=case.requested_sg,
        ))
        selected_candidates_by_graph[case.graph_id] = []

candidate_df = dataframe_result('algo20_pyxtal_candidates', candidate_rows)
candidate_df = candidate_df.sort_values(['graph', 'candidate_id']).reset_index(drop=True)
display(candidate_df)

scoring_df = dataframe_result('algo20_seed_scoring', scoring_rows)
scoring_df = scoring_df.sort_values(['graph', 'score_total', 'candidate_id']).reset_index(drop=True)
display(scoring_df)

selected_rows = []
for case in GRAPH_CASES:
    for rank_idx, candidate in enumerate(selected_candidates_by_graph.get(case.graph_id, []), start=1):
        score_row = next(row for row in scoring_rows if row.get('candidate_id') == candidate['candidate_id'] and row.get('status') != 'ERROR')
        selected_rows.append({
            'test': 'algo20_selected_seeds',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'rank': int(rank_idx),
            'candidate_id': candidate['candidate_id'],
            'score_total': float(score_row['score_total']),
            'probe_witness_sin': float(score_row['probe_witness_sin']),
            'probe_witness_rmse': float(score_row['probe_witness_rmse']),
            'min_pair_distance': float(score_row['min_pair_distance']),
            'template_self_sin': float(score_row['template_self_sin']),
            'PASS': True,
            'status': 'INFO',
        })

selected_df = dataframe_result('algo20_selected_seeds', selected_rows)
selected_df = selected_df.sort_values(['graph', 'rank']).reset_index(drop=True)
display(selected_df)


## 3. Seeded reverse sampling from the right basin

Now we stop asking a free prior sample to discover the oracle template.
We start from a **seed-compatible clean structure**, forward-noise it with the native KLDM/TDM process, and compare:

- baseline reverse from seed noise
- Algorithm 20 q-witness PPR reverse from the same seed noise

This is the clean seeded-basin proof target.


In [ ]:
def run_seeded_reverse(case: GraphCase, candidate, *, method: str, n_steps: int = SEEDED_FULL_STEPS, projection_interval: int = SEEDED_PROJECTION_INTERVAL, M: int = SEEDED_PPR_M, lambda0: float = SEEDED_PPR_LAMBDA0, tau: float = SEEDED_TAU, rollback: bool = SEEDED_ROLLBACK, seed: int = SAMPLE_SEED):
    graph_batch = single_graph_batch(case)
    init = runner.model._prepare_csp_sampling(
        batch=graph_batch,
        n_steps=int(n_steps),
        t_start=1.0,
        t_final=1e-6,
    )
    set_seed(int(seed) + 700000 * int(case.graph_id) + 1000 * int(hash(candidate['candidate_id']) % 997))
    t_start_nodes = torch.ones((int(case.atomic_numbers.numel()),), device=candidate['f0_model'].device, dtype=candidate['f0_model'].dtype)
    f_init, v_init, *_ = runner.model.tdm.sample_noisy_state(
        t=t_start_nodes,
        f0=candidate['f0_model'],
        index=init['node_index'],
    )
    state = {
        **init,
        'f_t': f_init.detach().clone(),
        'v_t': v_init.detach().clone(),
        'l_t': candidate['l0'].detach().clone().reshape(1, -1),
        'a_t': case.atomic_numbers.detach().clone(),
    }
    payload = candidate['payload']
    projection_rows = []
    accepted_count = 0
    feasible_count = 0
    projection_count = 0

    for step_idx, times in enumerate(iter_sampling_times(batch=state['batch'], grid=state['sampling_time_grid']), start=1):
        with torch.no_grad():
            preds_curr = runner.model.score_network(
                t=times.now.graph,
                pos=state['f_t'],
                v=state['v_t'],
                h=state['a_t'],
                l=state['l_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
            )
            state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_predictor(
                t=times.now.nodes,
                f_t=state['f_t'],
                v_t=state['v_t'],
                pred_v=preds_curr['v'],
                index=state['node_index'],
                dt=times.dt,
            )

        if method == 'ppr' and step_idx % int(projection_interval) == 0:
            projection_count += 1
            algo_state = make_algo_state_from_raw(
                f=state['f_t'],
                v=state['v_t'],
                l=state['l_t'].reshape(-1),
                atom_types=state['a_t'],
                node_index=state['node_index'],
                edge_node_index=state['edge_node_index'],
                t_graph=times.next.graph,
                t_nodes=times.next.nodes,
            )
            q_seed = make_payload_q_init(payload, mode='seed', seed=int(seed), graph_id=case.graph_id, dtype=state['f_t'].dtype, device=state['f_t'].device)
            kernel = ppr_kernel_q_witness(
                state=algo_state,
                payload=payload,
                model=runner.model,
                config=config20(M=int(M), lambda0=float(lambda0), q_init_mode='seed'),
                q_init=q_seed,
            )
            tail = kernel.logs[-1] if kernel.logs else {}
            feasible = bool(tail.get('soft_anchor_feasible', False))
            accepted = bool(feasible or not rollback)
            feasible_count += int(feasible)
            accepted_count += int(accepted)
            if accepted:
                state['f_t'] = kernel.state.f.detach().clone()
                state['v_t'] = kernel.state.v.detach().clone()
            projection_rows.append({
                'test': 'algo20_seeded_trace',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'candidate_id': candidate['candidate_id'],
                'method': 'seeded_ppr',
                'step': int(step_idx),
                't_next': float(times.t_next_float),
                'projection_interval': int(projection_interval),
                'M': int(M),
                'lambda0': float(lambda0),
                'rollback': bool(rollback),
                'c_after_witness_sin': float(tail.get('c_after_witness_sin', float('nan'))),
                'c_after_redenoise_witness': float(tail.get('c_after_redenoise_witness', float('nan'))),
                'witness_rmse_payload': float(tail.get('witness_rmse_payload', float('nan'))),
                'soft_anchor_feasible': feasible,
                'accepted': accepted,
                'q_norm': float(tail.get('q_norm', float('nan'))),
                'status': 'INFO',
                'PASS': True,
            })

        if times.t_next_float >= 1e-3:
            with torch.no_grad():
                preds_next = runner.model.score_network(
                    t=times.next.graph,
                    pos=state['f_t'],
                    v=state['v_t'],
                    h=state['a_t'],
                    l=state['l_t'],
                    node_index=state['node_index'],
                    edge_node_index=state['edge_node_index'],
                )
                state['f_t'], state['v_t'] = state['sampling_tdm'].reverse_step_corrector(
                    t=times.next.nodes,
                    f_t=state['f_t'],
                    v_t=state['v_t'],
                    pred_v=preds_next['v'],
                    dt=times.dt,
                    index=state['node_index'],
                    tau=float(tau),
                )
                state['l_t'] = runner.model._reverse_lattice_sampling_step(
                    t=times.next.lattice,
                    x_t=state['l_t'],
                    pred=preds_next['l'],
                    dt=times.dt,
                    num_atoms=state['batch'].num_atoms,
                )

    if state['restore_training']:
        state['score_network'].train()

    final_state = make_algo_state_from_raw(
        f=state['f_t'],
        v=state['v_t'],
        l=state['l_t'].reshape(-1),
        atom_types=state['a_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
        t_graph=torch.zeros((1, 1), device=state['f_t'].device, dtype=state['f_t'].dtype),
        t_nodes=torch.zeros((state['f_t'].shape[0],), device=state['f_t'].device, dtype=state['f_t'].dtype),
    )
    clean_f = clean_prediction(final_state)
    q_fit = q_only_fit_from_clean(clean_f, payload, graph_id=case.graph_id, q_init_mode='seed', seed=int(seed))
    eval_final = evaluate_arrays(case, clean_f, final_state.l, case.atomic_numbers)
    return {
        'test': 'algo20_seeded_compare',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'candidate_id': candidate['candidate_id'],
        'method': 'seeded_pc' if method == 'pc' else 'seeded_ppr',
        'n_steps': int(n_steps),
        'projection_interval': np.nan if method == 'pc' else float(projection_interval),
        'M': 0 if method == 'pc' else int(M),
        'lambda0': np.nan if method == 'pc' else float(lambda0),
        'rollback': False if method == 'pc' else bool(rollback),
        'final_witness_sin': float(q_fit['fit']['witness_sin']),
        'final_witness_rmse': float(q_fit['fit']['witness_rmse_payload']),
        'frac_rmse': float(eval_final['frac_rmse']),
        'rmse': float(eval_final['rmse']),
        'match': bool(eval_final['match']),
        'valid': bool(eval_final['valid']),
        'projection_count': np.nan if method == 'pc' else float(projection_count),
        'accepted_fraction': np.nan if method == 'pc' or projection_count == 0 else float(accepted_count / projection_count),
        'soft_anchor_feasible_fraction': np.nan if method == 'pc' or projection_count == 0 else float(feasible_count / projection_count),
        'PASS': True,
        'status': 'INFO',
    }, projection_rows


compare_rows = []
trace_rows = []
for case in GRAPH_CASES:
    for candidate in selected_candidates_by_graph.get(case.graph_id, []):
        try:
            pc_row, pc_trace = run_seeded_reverse(case, candidate, method='pc')
            compare_rows.append(pc_row)
            trace_rows.extend(pc_trace)
        except Exception as exc:
            compare_rows.append(error_row(
                'algo20_seeded_compare',
                case.graph_id,
                exc,
                'seeded_pc_reverse',
                space_group=case.requested_sg,
                candidate_id=candidate['candidate_id'],
                method='seeded_pc',
            ))
        try:
            ppr_row, ppr_trace = run_seeded_reverse(case, candidate, method='ppr')
            compare_rows.append(ppr_row)
            trace_rows.extend(ppr_trace)
        except Exception as exc:
            compare_rows.append(error_row(
                'algo20_seeded_compare',
                case.graph_id,
                exc,
                'seeded_ppr_reverse',
                space_group=case.requested_sg,
                candidate_id=candidate['candidate_id'],
                method='seeded_ppr',
            ))

compare_df = dataframe_result('algo20_seeded_compare', compare_rows)
compare_df = compare_df.sort_values(['graph', 'candidate_id', 'method']).reset_index(drop=True)
display(compare_df)

trace_df = dataframe_result('algo20_seeded_trace', trace_rows)
trace_df = trace_df.sort_values(['graph', 'candidate_id', 'step']).reset_index(drop=True) if not trace_df.empty else trace_df
display(trace_df)


## Notes

This notebook is intentionally **seeded** rather than free-prior.

If seeded PPR behaves well while free-prior PPR fails, that is still a meaningful result:

- it proves Algorithm 20 works as a local symmetry-constrained KLDM refiner
- it shows the remaining problem is basin discovery / template proposal, not the q-witness PPR kernel itself

Right now the payloads are still built from the **seed structure** because that is the cleanest way to reuse the existing DiffCSP++ transport bridge. That is acceptable for this seeded proof-of-mechanism notebook, even though the longer-term goal is a cleaner template-only representation.
